In [3]:
import os
import hashlib
import random
import json
from cryptography.fernet import Fernet

# A small fixed prime number and generator for demonstration
p = 10007
g = 2

user_data_file = "user_data.json"
encryption_key_file = "secret_key.key"

# Generate and store encryption key for secure storage
if not os.path.exists(encryption_key_file):
    encryption_key = Fernet.generate_key()
    with open(encryption_key_file, "wb") as key_file:
        key_file.write(encryption_key)
else:
    with open(encryption_key_file, "rb") as key_file:
        encryption_key = key_file.read()

cipher = Fernet(encryption_key)

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def encrypt_secret_key(secret_key):
    return cipher.encrypt(str(secret_key).encode()).decode()

def decrypt_secret_key(encrypted_key):
    return int(cipher.decrypt(encrypted_key.encode()).decode())

def register_user(username):
    print("\n--- User Registration ---")
    password = input("Create a password for your account: ")
    hashed_password = hash_password(password)
    secret_key = random.randint(1, p - 2)
    encrypted_key = encrypt_secret_key(secret_key)

    user_data = {}
    if os.path.exists(user_data_file):
        with open(user_data_file, "r") as f:
            user_data = json.load(f)

    user_data[username] = {
        "password": hashed_password,
        "secret_key": encrypted_key
    }

    with open(user_data_file, "w") as f:
        json.dump(user_data, f, indent=4)

    print("\n🎉 Registration Successful!")
    print("This application uses Zero-Knowledge Proof to verify you without revealing your secret.")
    print("Your unique secret key is:")
    print(f"🔑 Secret Key: {secret_key}")
    print("➡ Please save this key somewhere safe! It is needed for login.\n")

def get_secret_key(username):
    if not os.path.exists(user_data_file):
        print("⚠️ No users registered yet.")
        return None

    with open(user_data_file, "r") as f:
        user_data = json.load(f)

    if username in user_data:
        encrypted_key = user_data[username]["secret_key"]
        secret_key = decrypt_secret_key(encrypted_key)
        print(f"\n✅ Welcome back, {username}!")
        return secret_key

    print("❌ User not found. Please register first.")
    return None

def zero_knowledge_proof(secret_key):
    print("\n--- Zero Knowledge Proof Authentication ---")
    print("\n💡 In this process, you'll prove that you know your secret key WITHOUT revealing it.")
    print("We'll walk you through the steps...")

    # Step 1: Generate public key
    public_key = pow(g, secret_key, p)
    print(f"\nStep 1️⃣: Your public key is computed as g^secret_key mod p")
    print(f"Public Key (g^{secret_key} mod {p}) = {public_key}")

    # Step 2: Generate a random r and commitment
    r = random.randint(1, p - 2)
    commitment = pow(g, r, p)
    print(f"\nStep 2️⃣: A random value 'r' is chosen (kept secret), and commitment is computed.")
    print(f"Random r = {r}")
    print(f"Commitment = g^r mod p = {commitment}")

    # Step 3: Verifier generates a random challenge
    challenge = random.randint(1, p - 2)
    print(f"\nStep 3️⃣: A random challenge is issued by the verifier.")
    print(f"Challenge (c) = {challenge}")

    # Step 4: Prover calculates the response using the formula
    s = (r + challenge * secret_key) % (p - 1)
    print(f"\nStep 4️⃣: You (the prover) calculate the response using the formula:")
    print("s = (r + c × secret_key) mod (p - 1)")
    print(f"Response (s) = ({r} + {challenge} × {secret_key}) mod {p - 1} = {s}")

    # Step 5: Verifier checks if the proof is valid
    left = pow(g, s, p)
    right = (commitment * pow(public_key, challenge, p)) % p

    print(f"\nStep 5️⃣: Verifier checks if g^s ≡ commitment × (public_key)^c mod p")
    print(f"Left Side (g^s mod p): {left}")
    print(f"Right Side (commitment × public_key^c mod p): {right}")

    if left == right:
        print("\n✅ Success! Zero-Knowledge Proof verification passed.")
    else:
        print("\n❌ Verification failed. Something went wrong. Please try again.")

def main():
    print("🛡️ Welcome to the Zero-Knowledge Proof Demo App 🛡️")
    print(f"\n📘 We're using a fixed prime number p = {p} and generator g = {g}")
    print("🔐 This app demonstrates how you can authenticate without ever revealing your secret.")

    while True:
        print("\n📋 Main Menu")
        print("1. Register (First-time users)")
        print("2. Login & Prove Identity (ZKP)")
        print("3. Exit")
        choice = input("Choose an option (1/2/3): ")

        if choice == "1":
            username = input("\nEnter a username for registration: ")
            register_user(username)

        elif choice == "2":
            username = input("\nEnter your username to log in: ")
            secret_key = get_secret_key(username)
            if secret_key:
                zero_knowledge_proof(secret_key)

        elif choice == "3":
            print("\n👋 Thank you for using the ZKP Demo App. Goodbye!")
            break

        else:
            print("❗ Invalid choice. Please enter 1, 2, or 3.")

# Run the program
if __name__ == "__main__":
    main()


🛡️ Welcome to the Zero-Knowledge Proof Demo App 🛡️

📘 We're using a fixed prime number p = 10007 and generator g = 2
🔐 This app demonstrates how you can authenticate without ever revealing your secret.

📋 Main Menu
1. Register (First-time users)
2. Login & Prove Identity (ZKP)
3. Exit


Choose an option (1/2/3):  1

Enter a username for registration:  a



--- User Registration ---


Create a password for your account:  1



🎉 Registration Successful!
This application uses Zero-Knowledge Proof to verify you without revealing your secret.
Your unique secret key is:
🔑 Secret Key: 7922
➡ Please save this key somewhere safe! It is needed for login.


📋 Main Menu
1. Register (First-time users)
2. Login & Prove Identity (ZKP)
3. Exit


Choose an option (1/2/3):  2

Enter your username to log in:  a



✅ Welcome back, a!

--- Zero Knowledge Proof Authentication ---

💡 In this process, you'll prove that you know your secret key WITHOUT revealing it.
We'll walk you through the steps...

Step 1️⃣: Your public key is computed as g^secret_key mod p
Public Key (g^7922 mod 10007) = 4763

Step 2️⃣: A random value 'r' is chosen (kept secret), and commitment is computed.
Random r = 6836
Commitment = g^r mod p = 8567

Step 3️⃣: A random challenge is issued by the verifier.
Challenge (c) = 793

Step 4️⃣: You (the prover) calculate the response using the formula:
s = (r + c × secret_key) mod (p - 1)
Response (s) = (6836 + 793 × 7922) mod 10006 = 5214

Step 5️⃣: Verifier checks if g^s ≡ commitment × (public_key)^c mod p
Left Side (g^s mod p): 6433
Right Side (commitment × public_key^c mod p): 6433

✅ Success! Zero-Knowledge Proof verification passed.

📋 Main Menu
1. Register (First-time users)
2. Login & Prove Identity (ZKP)
3. Exit


Choose an option (1/2/3):  3



👋 Thank you for using the ZKP Demo App. Goodbye!
